## Pour l'essai 3, on va tester plusieurs nombre de PCA, et également plusieurs valeurs de régularisation (C) pour la régression logistique.

### Objectif : Améliorer la precision finale du modèle.



# Etapes précédentes

## Import divers

In [1]:
## Import via notre python source

import sys
from pathlib import Path

# Le noyau Jupyter peut démarrer à la racine du dépôt ou dans notebooks/.
racine = Path.cwd()
while not (racine / "src" / "donnees.py").is_file():
    if racine == racine.parent:
        raise FileNotFoundError("Ouvrez le notebook depuis le dépôt ML.")
    racine = racine.parent
if str(racine) not in sys.path:
    sys.path.insert(0, str(racine))

from src.donnees import creer_dataloaders

## Import du dataset

In [2]:
train_loader, val_loader, test_loader = creer_dataloaders(batch_size=64)

print(f"Train : {len(train_loader.dataset)} images")
print(f"Validation : {len(val_loader.dataset)} images")
print(f"Test : {len(test_loader.dataset)} images")

Train : 24402 images
Validation : 4307 images
Test : 7178 images


## Vérification des lots

In [3]:
print("Train :", len(train_loader))
print("Validation :", len(val_loader))
print("Test :", len(test_loader))

Train : 382
Validation : 68
Test : 113


## Applatissement des images et conservation des étiquettes :

In [4]:
import torch

def extraire_images_et_labels(loader):
    vecteurs = []
    etiquettes = []

    for images, labels in loader:
        vecteurs.append(images.flatten(start_dim=1))
        etiquettes.append(labels)

    X = torch.cat(vecteurs, dim=0).numpy()
    y = torch.cat(etiquettes, dim=0)
    return X, y

X_train, y_train = extraire_images_et_labels(train_loader)
X_val, y_val = extraire_images_et_labels(val_loader)

print(X_train.shape)  # (24402, 2304)
print(X_val.shape)    # (4307, 2304)

(24402, 2304)
(4307, 2304)


# Nouvelle expérience

In [5]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("pca", PCA(svd_solver="randomized", random_state=42)),
    ("logreg", LogisticRegression(solver="lbfgs", max_iter=1000)),
])

recherche = GridSearchCV(
    pipeline,
    param_grid={
        "pca__n_components": [100, 150, 200],
        "logreg__C": [0.1, 1, 3, 10],
    },
    cv=3,
    scoring="accuracy",
    n_jobs=1,
)

recherche.fit(X_train, y_train)
print("Meilleurs paramètres :", recherche.best_params_)
print(f"Score sur ta validation : {recherche.score(X_val, y_val):.2%}")

Meilleurs paramètres : {'logreg__C': 0.1, 'pca__n_components': 150}
Score sur ta validation : 37.75%
